In [2]:
import pandas as pd

# -----------------------------
# 1. CREATE INTER-ARRIVAL RDA TABLE
# -----------------------------
def create_rda_table(times, probabilities):
    c_probs = []
    current_sum = 0
    for p in probabilities:
        current_sum += p
        c_probs.append(round(current_sum, 3))

    ranges = []
    start = 0
    for cp in c_probs:
        end = int(cp * 1000)
        ranges.append(f"{start:03d}-{end:03d}")
        start = end + 1

    df = pd.DataFrame({
        'Time': times,
        'Probability': probabilities,
        'Cumulative Prob': c_probs,
        'Random Digit Range': ranges
    })
    return df


inter_arrival_times = [1, 2, 3, 4, 5, 6, 7, 8]
inter_arrival_probs = [0.125] * 8

inter_arrival_table = create_rda_table(inter_arrival_times, inter_arrival_probs)


# -----------------------------
# 2. CREATE SERVICE TIME RDA TABLE
# -----------------------------
def assign_service_rda(service_times, service_probs):
    c_probs = []
    current_total = 0
    for p in service_probs:
        current_total += p
        c_probs.append(round(current_total, 2))

    rda_ranges = []
    start_digit = 0
    for cp in c_probs:
        end_digit = int(cp * 100)
        if end_digit == 100:
            rda_ranges.append(f"{start_digit:02d} - {end_digit:03d}")
        else:
            rda_ranges.append(f"{start_digit:02d} - {end_digit:02d}")
        start_digit = end_digit + 1

    return pd.DataFrame({
        'Service Time (min)': service_times,
        'Probability': service_probs,
        'Cumulative Prob': c_probs,
        'Random Digit Range': rda_ranges
    })


s_times = [1, 2, 3, 4, 5, 6]
s_probs = [0.10, 0.20, 0.30, 0.25, 0.10, 0.05]

service_table = assign_service_rda(s_times, s_probs)


# -----------------------------
# 3. INTER-ARRIVAL SIMULATION
# -----------------------------
def get_inter_arrival_time(random_digit, rda_table):
    for _, row in rda_table.iterrows():
        start_str, end_str = row['Random Digit Range'].split('-')
        start, end = int(start_str), int(end_str)
        if start <= random_digit <= end:
            return row['Time']
    return None


initial_customer_row = pd.DataFrame({
    'Random Digit': [0],
    'Inter-Arrival Time (min)': [0]
})

arrival_random_digits = [161, 658, 359, 606, 198]

simulated_arrivals = []
for rd in arrival_random_digits:
    ia_time = get_inter_arrival_time(rd, inter_arrival_table)
    simulated_arrivals.append({
        'Random Digit': rd,
        'Inter-Arrival Time (min)': ia_time
    })

subsequent_arrivals_df = pd.DataFrame(simulated_arrivals)
final_inter_arrival_df = pd.concat(
    [initial_customer_row, subsequent_arrivals_df],
    ignore_index=True
)


# -----------------------------
# 4. SERVICE TIME SIMULATION
# -----------------------------
def get_service_time(random_digit, rda_table):
    for _, row in rda_table.iterrows():
        start_str, end_str = row['Random Digit Range'].split(' - ')
        start, end = int(start_str), int(end_str)
        if start <= random_digit <= end:
            return row['Service Time (min)']
    return None


service_random_digits = [66, 97, 89, 2, 16, 70]

simulated_service = []
for rd in service_random_digits:
    service_time = get_service_time(rd, service_table)
    simulated_service.append({
        'Random Digit': rd,
        'Service Time (min)': service_time
    })

simulated_service_df = pd.DataFrame(simulated_service)


# -----------------------------
# 5. FULL QUEUE SIMULATION TABLE
# -----------------------------
simulation_df = pd.DataFrame({
    'Inter-Arrival Time (min)': final_inter_arrival_df['Inter-Arrival Time (min)'],
    'Service Time (min)': simulated_service_df['Service Time (min)']
})

arrival_times = []
service_start_times = []
service_end_times = []
wait_times = []
total_times_in_system = []
idle_times = []

prev_arrival_time = 0
prev_service_end_time = 0

for index, row in simulation_df.iterrows():
    inter_arrival_time = row['Inter-Arrival Time (min)']
    service_time = row['Service Time (min)']

    if index == 0:
        current_arrival_time = 0
        current_service_begin_time = current_arrival_time
        current_service_end_time = current_service_begin_time + service_time
        current_wait_time = 0
        current_total_time = current_service_end_time - current_arrival_time
        current_idle_time = 0
    else:
        current_arrival_time = prev_arrival_time + inter_arrival_time
        current_service_begin_time = max(current_arrival_time, prev_service_end_time)
        current_service_end_time = current_service_begin_time + service_time
        current_wait_time = current_service_begin_time - current_arrival_time
        current_total_time = current_service_end_time - current_arrival_time
        current_idle_time = max(0, current_service_begin_time - prev_service_end_time)

    arrival_times.append(current_arrival_time)
    service_start_times.append(current_service_begin_time)
    service_end_times.append(current_service_end_time)
    wait_times.append(current_wait_time)
    total_times_in_system.append(current_total_time)
    idle_times.append(current_idle_time)

    prev_arrival_time = current_arrival_time
    prev_service_end_time = current_service_end_time

simulation_df['Arrival Time'] = arrival_times
simulation_df['Time Service Begin'] = service_start_times
simulation_df['Time Service End'] = service_end_times
simulation_df['Waiting Time'] = wait_times
simulation_df['Total Time in System'] = total_times_in_system
simulation_df['Idle Time'] = idle_times


# -----------------------------
# DISPLAY ALL 5 TABLES
# -----------------------------
display(inter_arrival_table)
display(service_table)
display(final_inter_arrival_df)
display(simulated_service_df)
display(simulation_df)

,Time,Probability,Cumulative Prob,Random Digit Range
0,1,0.125,0.125,000-125
1,2,0.125,0.250,126-250
2,3,0.125,0.375,251-375
3,4,0.125,0.500,376-500
4,5,0.125,0.625,501-625
5,6,0.125,0.750,626-750
6,7,0.125,0.875,751-875
7,8,0.125,1.000,876-1000


,Service Time (min),Probability,Cumulative Prob,Random Digit Range
0,1,0.10,0.10,00 - 10
1,2,0.20,0.30,11 - 30
2,3,0.30,0.60,31 - 60
3,4,0.25,0.85,61 - 85
4,5,0.10,0.95,86 - 95
5,6,0.05,1.00,96 - 100


,Random Digit,Inter-Arrival Time (min)
0,0,0
1,161,2
2,658,6
3,359,3
4,606,5
5,198,2


,Random Digit,Service Time (min)
0,66,4
1,97,6
2,89,5
3,2,1
4,16,2
5,70,4


,Inter-Arrival Time (min),Service Time (min),Arrival Time,Time Service Begin,Time Service End,Waiting Time,Total Time in System,Idle Time
0,0,4,0,0,4,0,4,0
1,2,6,2,4,10,2,8,0
2,6,5,8,10,15,2,7,0
3,3,1,11,15,16,4,5,0
4,5,2,16,16,18,0,2,0
5,2,4,18,18,22,0,4,0
